<div style="font-family: 'Helvetica Neue', Arial, sans-serif; background:#fff; padding: 36px 40px; border-radius: 8px; margin-bottom: 4px; position:relative; overflow:hidden; border: 1.5px solid #e8e8e8;">

  <div style="font-size:130px; font-weight:900; color:rgba(0,0,0,0.04); position:absolute; top:-20px; right:30px; line-height:1; letter-spacing:-0.05em;">02</div>

  <div style="font-size:10px; color:#00D563; letter-spacing:0.25em; text-transform:uppercase; margin-bottom:16px;">NOVA IMS &middot; 2025/2026</div>
  <div style="font-size:36px; font-weight:800; color:#111; letter-spacing:-0.02em; line-height:1.1; margin-bottom:6px;">One Pipeline, <span style="color:#00D563;">One Model.</span></div>
  <div style="font-size:12px; color:#00D563; font-weight:500; margin-bottom:24px;">Notebook 2 &mdash; Final Solution &middot; Restart &amp; Run All</div>

  <div style="display:flex; gap:48px;">
    <div>
      <div style="font-size:9px; color:#00D563; letter-spacing:0.2em; text-transform:uppercase; margin-bottom:6px;">Group 33</div>
      <div style="font-size:11px; color:#555; line-height:1.9;">Alexandra Varela, 20250514<br>Francisca Fernandes, 20250406<br>Mariana Melo, 20250414<br>Tiago Antunes, 20250357</div>
    </div>
    <div>
      <div style="font-size:9px; color:#00D563; letter-spacing:0.2em; text-transform:uppercase; margin-bottom:6px;">Course</div>
      <div style="font-size:11px; color:#555; line-height:1.9;">Text Mining<br>MSc Data Science &amp; Advanced Analytics<br>NOVA Information Management School</div>
    </div>
  </div>
</div>


**Submitted solution - an integrated weighted soft-voting ensemble pipeline.**
The instructor confirmed (e-mail, June 2026) that an ensemble is an acceptable final solution
provided it is *a single, integrated end-to-end pipeline that consumes the input data in a unified
way through to the final prediction*. This notebook is exactly that: a weighted soft-vote over
**8 fine-tuned transformer encoders**. If the submitted ensemble JSON is present, the notebook
loads those weights; if it is absent, it trains/loads all 8 members and optimizes the weights from
leak-free OOF probabilities inside the notebook.

- **Pipeline:** raw tweets -> `fix_tweet` preprocessing -> 8 encoders -> OOF weight optimization -> weighted soft-vote -> label.
- **All 8 encoders** are notebook-native load-or-train members: if their OOF/test probability caches are present, the notebook loads them; if they are absent, the notebook trains the missing member(s) from scratch and recreates the caches/results.
- **No embedded result constants:** without `results/tables/ensemble_optimal_result.json`, weights are computed from OOF probabilities rather than copied from a hidden JSON blob.
- **Preprocessing:** `fix_tweet` - `ftfy` mojibake repair + truncation-artefact/URL cleanup (alters 50.1% of tweets).
- **Protocol:** 10-fold stratified CV, seed 42, best-checkpoint-per-fold, fp16 + GradScaler, cosine LR warmup, class-weighted loss + label smoothing.

| Configuration | OOF F1-macro |
|---|---|
| Best individual encoder (FinBERT 10ep fix_text) | 0.9082 |
| Distilled single model (extra work, 12ep) | 0.9139 |
| **Weighted soft-vote ensemble, 8 encoders (SUBMITTED)** | **0.9201** |

The ensemble gain over the best individual encoder is statistically significant (paired bootstrap 95% CI of the F1 difference: [+0.0066, +0.0172], n=1000). **Runtime:** ~2 min with committed probabilities and JSON; if caches are absent, the notebook trains the missing encoders and then optimizes the ensemble, which can take hours on a CUDA GPU. **Instructions:** Kernel -> Restart Kernel and Run All Cells. Produces `pred_33.csv`.


In [1]:
# Cell 1: Imports and reproducibility
import os, sys, re, time, gc, json, warnings, tempfile
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Auto-detect the notebook root: the folder that contains data/raw/train.csv.
# Ensures all relative paths work regardless of where Jupyter was launched from.
def _find_nb_root():
    cwd = os.getcwd()
    for candidate in [cwd,
                      os.path.join(cwd, 'deliverables', 'group_33'),
                      os.path.join(cwd, 'group_33')]:
        if os.path.exists(os.path.join(candidate, 'data', 'raw', 'train.csv')):
            return os.path.abspath(candidate)
    p = cwd
    for _ in range(3):
        p = os.path.dirname(p)
        if os.path.exists(os.path.join(p, 'data', 'raw', 'train.csv')):
            return os.path.abspath(p)
    return None

_nb_root = _find_nb_root()
if _nb_root:
    os.chdir(_nb_root)
    print(f'Working directory: {_nb_root}')
else:
    print(f'WARNING: data/raw/train.csv not found near {os.getcwd()}')
    print('Place train.csv and test.csv in data/raw/ before running.')


import ftfy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, accuracy_score, precision_score,
                             recall_score, classification_report)
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          get_cosine_schedule_with_warmup)

SEED = 42

def seed_all(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_all()
print(f'Seed fixed: {SEED}')
print(f'Python: {sys.version[:20]} | torch {torch.__version__}')
print(f'ftfy {ftfy.__version__}')

Working directory: c:\Users\tiago\OneDrive - NOVAIMS\Ambiente de Trabalho\textmining\group_33\deliverables\group_33
Seed fixed: 42
Python: 3.11.9 (tags/v3.11.9 | torch 2.12.0+cu130
ftfy 6.3.1


In [2]:
# Cell 3: Preprocessing — ftfy mojibake fix + truncation cleanup
# Text-quality audit of the source CSV: 7.0% of tweets carry UTF-8 mojibake
# (e.g. â€™ instead of '), 16.4% end in truncation artefacts and 46.8% contain
# URLs with no sentiment value. fix_tweet repairs/removes these (alters 50.1%
# of tweets). Ablation at 10-fold: +0.26pp OOF F1-macro on FinBERT vs raw text.

_TRUNC_RE = re.compile(r'[�…°�…]+\s*(https?://\S*)?$')
_URL_RE    = re.compile(r'https?://\S+')
_TRAIL_RE  = re.compile(r'[\s\-–:]+$')

def fix_tweet(text: str) -> str:
    """Fix mojibake (ftfy) and remove truncation artefacts + bare URLs."""
    text = ftfy.fix_text(str(text))
    text = _TRUNC_RE.sub('', text)
    text = _URL_RE.sub('', text)
    return _TRAIL_RE.sub('', text).strip()

train = pd.read_csv('data/raw/train.csv')
test  = pd.read_csv('data/raw/test.csv')

raw_texts       = train['text'].astype(str).tolist()
raw_test_texts  = test['text'].astype(str).tolist()
fixed_texts     = [fix_tweet(t) for t in raw_texts]
fixed_test_texts= [fix_tweet(t) for t in raw_test_texts]

# Default names keep downstream cells readable; per-model configs choose raw vs fixed.
texts      = fixed_texts
test_texts = fixed_test_texts
y = train['label'].values

print(f'Train: {train.shape} | Test: {test.shape}')
print('Label distribution (0=Bearish, 1=Bullish, 2=Neutral):')
print(train['label'].value_counts().sort_index())
print(f'\nSample before: {train["text"].iloc[1][:90]}')
print(f'Sample after:  {texts[1][:90]}')

Train: (9543, 2) | Test: (2388, 2)
Label distribution (0=Bearish, 1=Bullish, 2=Neutral):
label
0    1442
1    1923
2    6178
Name: count, dtype: int64

Sample before: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.c
Sample after:  $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean


In [3]:

# Cell 4: Configuration - notebook-native weighted soft-voting ensemble
# CANDIDATE_TAGS: every model considered for the ensemble.
# When ensemble_optimal_result.json is absent, the notebook:
#   (1) loads or trains each candidate,
#   (2) ranks by individual OOF F1-macro,
#   (3) keeps only those >= MIN_OOF_F1,
#   (4) runs an exhaustive weight grid search over the survivors,
#   (5) writes the result to ensemble_optimal_result.json.
# Selection and weighting are fully data-driven — no constants embedded.

PRIMARY_TAG  = 'finbert_fintwitter_10ep_fixtext'
N_FOLDS      = 10
MIN_OOF_F1   = 0.87   # minimum individual OOF F1-macro to enter the ensemble

CANDIDATE_TAGS = [
    'finbert_fintwitter_10ep_fixtext',
    'finbert_fintwitter_7ep',
    'roberta_large_ts_v2',
    'debertav3_large_6ep',
    'finbert_fintwitter_10ep',
    'finbert_fintwitter',
    'debertav3_large_6ep_fixtext',
    'deberta_base_finance_fixtext',
]

def _config_from_result_json(tag):
    """Reconstruct training config from results/tables/{tag}_result.json."""
    p = Path(f'results/tables/{tag}_result.json')
    if not p.exists():
        return None
    data = json.loads(p.read_text(encoding='utf-8'))
    r = data.get('recipe', {})
    return {
        'model_name':      data['model_id'],
        'epochs':          r['epochs'],
        'maxlen':          r['maxlen'],
        'lr':              r['lr'],
        'weight_decay':    r['weight_decay'],
        'batch_size':      r['batch_size'],
        'grad_accum':      r.get('grad_accum', 1),
        'label_smoothing': r['label_smoothing'],
        'warmup_ratio':    r['warmup_ratio'],
        'llrd':            r.get('llrd', 0.0),
        'schedule':        r.get('schedule', 'cosine'),
        'amp_dtype':       r.get('amp_dtype', 'fp16'),
        'eval_batch_size': r.get('eval_batch_size', 32),
        'fix_text':        r.get('fix_text', False),
    }

# Fallback configs — used when results/tables/{tag}_result.json is absent
# (i.e. running from scratch with only train.csv + test.csv).
# Values are the exact hyperparameters used in the submitted runs.
DEFAULT_CONFIGS = {
    'finbert_fintwitter_10ep_fixtext': {
        'model_name': 'nickmuchi/finbert-tone-finetuned-fintwitter-classification',
        'epochs': 10, 'maxlen': 128, 'lr': 5e-6, 'weight_decay': 0.01,
        'batch_size': 16, 'grad_accum': 1, 'label_smoothing': 0.05,
        'warmup_ratio': 0.06, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': True,
    },
    'finbert_fintwitter_7ep': {
        'model_name': 'nickmuchi/finbert-tone-finetuned-fintwitter-classification',
        'epochs': 7, 'maxlen': 128, 'lr': 8e-6, 'weight_decay': 0.01,
        'batch_size': 16, 'grad_accum': 1, 'label_smoothing': 0.05,
        'warmup_ratio': 0.06, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': False,
    },
    'roberta_large_ts_v2': {
        'model_name': 'cardiffnlp/twitter-roberta-large-topic-sentiment-latest',
        'epochs': 5, 'maxlen': 128, 'lr': 1e-5, 'weight_decay': 0.01,
        'batch_size': 16, 'grad_accum': 1, 'label_smoothing': 0.05,
        'warmup_ratio': 0.06, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': False,
    },
    'debertav3_large_6ep': {
        'model_name': 'microsoft/deberta-v3-large',
        'epochs': 6, 'maxlen': 128, 'lr': 1e-5, 'weight_decay': 0.01,
        'batch_size': 8, 'grad_accum': 4, 'label_smoothing': 0.05,
        'warmup_ratio': 0.1, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': False,
    },
    'finbert_fintwitter_10ep': {
        'model_name': 'nickmuchi/finbert-tone-finetuned-fintwitter-classification',
        'epochs': 10, 'maxlen': 128, 'lr': 5e-6, 'weight_decay': 0.01,
        'batch_size': 16, 'grad_accum': 1, 'label_smoothing': 0.05,
        'warmup_ratio': 0.06, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': False,
    },
    'finbert_fintwitter': {
        'model_name': 'nickmuchi/finbert-tone-finetuned-fintwitter-classification',
        'epochs': 5, 'maxlen': 128, 'lr': 1e-5, 'weight_decay': 0.01,
        'batch_size': 16, 'grad_accum': 1, 'label_smoothing': 0.05,
        'warmup_ratio': 0.06, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': False,
    },
    'debertav3_large_6ep_fixtext': {
        'model_name': 'microsoft/deberta-v3-large',
        'epochs': 6, 'maxlen': 128, 'lr': 1e-5, 'weight_decay': 0.01,
        'batch_size': 8, 'grad_accum': 4, 'label_smoothing': 0.05,
        'warmup_ratio': 0.1, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': True,
    },
    'deberta_base_finance_fixtext': {
        'model_name': 'nickmuchi/deberta-v3-base-finetuned-finance-text-classification',
        'epochs': 8, 'maxlen': 128, 'lr': 1e-5, 'weight_decay': 0.01,
        'batch_size': 16, 'grad_accum': 2, 'label_smoothing': 0.05,
        'warmup_ratio': 0.1, 'llrd': 0.0, 'schedule': 'cosine',
        'amp_dtype': 'fp16', 'eval_batch_size': 32, 'fix_text': True,
    },
}

MODEL_CONFIGS = {}
for _tag in CANDIDATE_TAGS:
    _cfg = _config_from_result_json(_tag)
    if _cfg:
        MODEL_CONFIGS[_tag] = _cfg
    elif _tag in DEFAULT_CONFIGS:
        MODEL_CONFIGS[_tag] = DEFAULT_CONFIGS[_tag]
        print(f'[DEFAULT CFG] {_tag} — result JSON absent, using hardcoded hyperparameters.')
    else:
        print(f'[WARN] No config for {_tag} — will fail if cache is also missing.')

ensemble_path = Path('results/tables/ensemble_optimal_result.json')
if ensemble_path.exists():
    ENSEMBLE = json.loads(ensemble_path.read_text(encoding='utf-8'))
    ENSEMBLE_WEIGHTS = ENSEMBLE['models']
    print(f'Loaded ensemble weights  (OOF F1-macro: {ENSEMBLE["oof_f1_macro"]:.4f})')
else:
    ENSEMBLE = None
    ENSEMBLE_WEIGHTS = None
    print(f'No ensemble JSON found — members will be auto-selected '
          f'(OOF F1 >= {MIN_OOF_F1}) after load/train.')

print(f'Candidates ({len(CANDIDATE_TAGS)}):')
for t in CANDIDATE_TAGS:
    flag = '  <- primary' if t == PRIMARY_TAG else ''
    w = ('' if ENSEMBLE_WEIGHTS is None or t not in ENSEMBLE_WEIGHTS
         else f' (w={ENSEMBLE_WEIGHTS[t]})')
    print(f'  {t}{w}{flag}')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    print(f'GPU: {torch.cuda.get_device_name(0)} | '
          f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    torch.set_num_threads(min(12, os.cpu_count() or 1))
    print('CPU mode — training missing candidates from scratch will be slow.')

counts = np.bincount(y, minlength=3)
class_weights = torch.tensor(len(y) / (3 * counts), dtype=torch.float32)
print(f'class weights: {[round(x,3) for x in class_weights.tolist()]}')


Loaded ensemble weights  (OOF F1-macro: 0.9201)
Candidates (8):
  finbert_fintwitter_10ep_fixtext (w=1.0)  <- primary
  finbert_fintwitter_7ep (w=1.0)
  roberta_large_ts_v2 (w=1.0)
  debertav3_large_6ep (w=1.0)
  finbert_fintwitter_10ep (w=0.75)
  finbert_fintwitter (w=0.75)
  debertav3_large_6ep_fixtext (w=0.75)
  deberta_base_finance_fixtext (w=0.5)
GPU: NVIDIA GeForce RTX 5070 | VRAM 12.8 GB
class weights: [2.206, 1.654, 0.515]


In [4]:
# Cell 5: General model builder, training loop and inference helpers
# These helpers mirror scripts/run_transformer_cv.py, but live in the notebook so
# a JSON/probability-cache-free run can still train every missing ensemble member.

def _amp_dtype_from_config(cfg):
    if DEVICE != 'cuda':
        return None
    if cfg.get('amp_dtype') == 'fp16':
        return torch.float16
    if cfg.get('amp_dtype') == 'bf16':
        return torch.bfloat16
    return None


def build_model(model_name):
    return AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=3, ignore_mismatched_sizes=True,
        dtype=torch.float32,   # fp32 params required for GradScaler
    ).to(DEVICE)


def build_optimizer_with_llrd(model, base_lr, weight_decay, llrd):
    no_decay = {'bias', 'LayerNorm.weight', 'layer_norm.weight'}
    num_layers = getattr(model.config, 'num_hidden_layers', None)
    if llrd == 0.0 or num_layers is None:
        decay = [p for n, p in model.named_parameters() if p.requires_grad and not any(nd in n for nd in no_decay)]
        no_decay_p = [p for n, p in model.named_parameters() if p.requires_grad and any(nd in n for nd in no_decay)]
        return torch.optim.AdamW([
            {'params': decay, 'lr': base_lr, 'weight_decay': weight_decay},
            {'params': no_decay_p, 'lr': base_lr, 'weight_decay': 0.0},
        ], lr=base_lr)

    groups = []
    for layer_idx in range(num_layers):
        layer_lr = base_lr * (llrd ** (num_layers - 1 - layer_idx))
        layer_prefix = f'encoder.layer.{layer_idx}.'
        decay = [p for n, p in model.named_parameters()
                 if p.requires_grad and layer_prefix in n and not any(nd in n for nd in no_decay)]
        no_decay_p = [p for n, p in model.named_parameters()
                      if p.requires_grad and layer_prefix in n and any(nd in n for nd in no_decay)]
        if decay:
            groups.append({'params': decay, 'lr': layer_lr, 'weight_decay': weight_decay})
        if no_decay_p:
            groups.append({'params': no_decay_p, 'lr': layer_lr, 'weight_decay': 0.0})

    other = [p for n, p in model.named_parameters()
             if p.requires_grad and not any(f'encoder.layer.{i}.' in n for i in range(num_layers))]
    if other:
        groups.append({'params': other, 'lr': base_lr, 'weight_decay': weight_decay})
    return torch.optim.AdamW(groups, lr=base_lr)


@torch.no_grad()
def predict_proba(model, tokenizer, txts, maxlen, eval_batch_size, amp_dtype):
    model.eval()
    out = []
    for i in range(0, len(txts), eval_batch_size):
        enc = tokenizer(txts[i:i+eval_batch_size], padding=True, truncation=True,
                        max_length=maxlen, return_tensors='pt')
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        if DEVICE == 'cuda' and amp_dtype is not None:
            with torch.autocast(device_type='cuda', dtype=amp_dtype):
                logits = model(**enc).logits
        else:
            logits = model(**enc).logits
        out.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.vstack(out)


def run_fold(cfg, tokenizer, train_texts, tr_idx, va_idx, all_test_texts, fold):
    seed_all(SEED + fold)
    amp_dtype = _amp_dtype_from_config(cfg)
    batch_size = cfg['batch_size']
    grad_accum = cfg.get('grad_accum', 1)
    eval_batch_size = cfg.get('eval_batch_size', 32)

    model = build_model(cfg['model_name'])
    optimizer = build_optimizer_with_llrd(model, cfg['lr'], cfg['weight_decay'], cfg.get('llrd', 0.0))
    steps_per_epoch = int(np.ceil(len(tr_idx) / batch_size))
    total_steps = max(1, (steps_per_epoch // grad_accum) * cfg['epochs'])
    warmup_steps = int(cfg['warmup_ratio'] * total_steps)
    scheduler = get_cosine_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    ce_loss = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE), label_smoothing=cfg['label_smoothing'])
    use_scaler = DEVICE == 'cuda' and amp_dtype == torch.float16
    scaler = torch.amp.GradScaler('cuda') if use_scaler else None

    fold_texts = [train_texts[i] for i in tr_idx]
    fold_labels = y[tr_idx]
    valid_texts = [train_texts[i] for i in va_idx]

    best_f1, best_va_proba = -1.0, None
    with tempfile.TemporaryDirectory() as tmpdir:
        ckpt = Path(tmpdir) / 'best.pt'
        for epoch in range(cfg['epochs']):
            model.train()
            order = np.random.permutation(len(fold_texts))
            running, t0 = 0.0, time.time()
            optimizer.zero_grad(set_to_none=True)
            for step, start in enumerate(range(0, len(order), batch_size)):
                bidx = order[start:start+batch_size]
                bt = [fold_texts[j] for j in bidx]
                bl = torch.tensor(fold_labels[bidx], dtype=torch.long, device=DEVICE)
                enc = tokenizer(bt, padding=True, truncation=True, max_length=cfg['maxlen'], return_tensors='pt')
                enc = {k: v.to(DEVICE) for k, v in enc.items()}
                if DEVICE == 'cuda' and amp_dtype is not None:
                    with torch.autocast(device_type='cuda', dtype=amp_dtype):
                        loss = ce_loss(model(**enc).logits, bl)
                else:
                    loss = ce_loss(model(**enc).logits, bl)

                loss = loss / grad_accum
                if use_scaler:
                    scaler.scale(loss).backward()
                else:
                    loss.backward()
                running += float(loss.item()) * grad_accum

                if (step + 1) % grad_accum == 0 or (start + batch_size) >= len(order):
                    if use_scaler:
                        scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.0)
                    if use_scaler:
                        scaler.step(optimizer); scaler.update()
                    else:
                        optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad(set_to_none=True)

            va_proba = predict_proba(model, tokenizer, valid_texts, cfg['maxlen'], eval_batch_size, amp_dtype)
            ep_f1 = f1_score(y[va_idx], va_proba.argmax(1), average='macro')
            star = ' *** best ***' if ep_f1 > best_f1 else ''
            print(f"    fold {fold}/{cfg.get('n_folds', N_FOLDS)} epoch {epoch+1}/{cfg['epochs']} "
                  f"loss={running/steps_per_epoch:.4f} val_f1={ep_f1:.6f} ({time.time()-t0:.0f}s){star}")
            if ep_f1 > best_f1:
                best_f1 = ep_f1
                best_va_proba = va_proba.copy()
                torch.save(model.state_dict(), ckpt)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
        best_test_proba = predict_proba(model, tokenizer, all_test_texts, cfg['maxlen'], eval_batch_size, amp_dtype)

    del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return best_va_proba, best_test_proba, best_f1


def train_member_from_scratch(tag, cfg):
    print(f"[TRAINING] {tag} - {cfg['epochs']} epochs, {N_FOLDS}-fold, model={cfg['model_name']}")
    train_texts = fixed_texts if cfg.get('fix_text') else raw_texts
    all_test_texts = fixed_test_texts if cfg.get('fix_text') else raw_test_texts
    tokenizer = AutoTokenizer.from_pretrained(cfg['model_name'])
    cfg = {**cfg, 'n_folds': N_FOLDS}

    cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    oof = np.zeros((len(y), 3), np.float32)
    tsum = np.zeros((len(all_test_texts), 3), np.float32)
    fold_scores = []
    started = time.time()
    for fold, (tr_idx, va_idx) in enumerate(cv.split(train_texts, y), start=1):
        print()
        print(f'=== {tag} fold {fold}/{N_FOLDS} ===')
        vp, tp, best = run_fold(cfg, tokenizer, train_texts, tr_idx, va_idx, all_test_texts, fold)
        oof[va_idx] = vp
        tsum += tp
        fold_scores.append(float(best))

    test_proba = (tsum / N_FOLDS).astype(np.float32)
    oof_pred = oof.argmax(axis=1)
    test_pred = test_proba.argmax(axis=1)
    result = {
        'tag': tag,
        'model_id': cfg['model_name'],
        'F1-macro': float(f1_score(y, oof_pred, average='macro')),
        'F1-macro_std': float(np.std(fold_scores)),
        'F1-weighted': float(f1_score(y, oof_pred, average='weighted')),
        'Accuracy': float(accuracy_score(y, oof_pred)),
        'Precision-macro': float(precision_score(y, oof_pred, average='macro', zero_division=0)),
        'Recall-macro': float(recall_score(y, oof_pred, average='macro', zero_division=0)),
        'per_fold_f1': fold_scores,
        'elapsed_min': float((time.time() - started) / 60),
        'test_dist': np.bincount(test_pred, minlength=3).astype(int).tolist(),
        'recipe': {
            'epochs': cfg['epochs'], 'maxlen': cfg['maxlen'], 'lr': cfg['lr'],
            'weight_decay': cfg['weight_decay'], 'batch_size': cfg['batch_size'],
            'effective_batch_size': cfg['batch_size'] * cfg.get('grad_accum', 1),
            'grad_accum': cfg.get('grad_accum', 1), 'label_smoothing': cfg['label_smoothing'],
            'warmup_ratio': cfg['warmup_ratio'], 'llrd': cfg.get('llrd', 0.0),
            'schedule': cfg.get('schedule', 'cosine'), 'amp_dtype': cfg.get('amp_dtype', 'fp16'),
            'eval_batch_size': cfg.get('eval_batch_size', 32), 'n_folds': N_FOLDS,
            'class_weighted_loss': True, 'best_checkpoint_per_fold': True,
            'fix_text': bool(cfg.get('fix_text')),
        },
    }

    Path('results/predictions').mkdir(parents=True, exist_ok=True)
    Path('results/tables').mkdir(parents=True, exist_ok=True)
    np.save(f'results/predictions/oof_proba_{tag}.npy', oof)
    np.save(f'results/predictions/test_proba_{tag}.npy', test_proba)
    pd.DataFrame(test_proba, columns=['p0','p1','p2']).to_csv(f'results/predictions/prob_test_{tag}.csv', index=False)
    pd.DataFrame({'id': test['id'], 'label': test_pred}).to_csv(f'results/predictions/pred_{tag}.csv', index=False)
    Path(f'results/tables/{tag}_result.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
    print(f"[DONE] {tag}: OOF F1={result['F1-macro']:.4f}; caches/results saved.")
    return test_proba, oof



def _macro_f1_for_prediction_batch(preds, labels):
    labels = np.asarray(labels)
    f1s = []
    for cls in range(3):
        pred_cls = preds == cls
        true_cls = labels[None, :] == cls
        tp = (pred_cls & true_cls).sum(axis=1).astype(float)
        fp = (pred_cls & ~true_cls).sum(axis=1).astype(float)
        fn = (~pred_cls & true_cls).sum(axis=1).astype(float)
        denom = 2 * tp + fp + fn
        f1s.append(np.divide(2 * tp, denom, out=np.zeros_like(tp), where=denom > 0))
    return np.mean(np.vstack(f1s), axis=0)


def optimize_ensemble_from_oof(member_oof, labels, tags, grid=(0.0, 0.25, 0.5, 0.75, 1.0), batch_size=256):
    """Run an OOF grid search when ensemble_optimal_result.json is missing."""
    from itertools import product

    tags = [tag for tag in tags if tag in member_oof]
    if not tags:
        raise ValueError('No OOF probabilities available for ensemble optimisation.')

    print()
    print('Optimizing ensemble weights from OOF probabilities (no JSON fallback).')
    print(f'Grid: {grid}; members: {len(tags)}; batch_size={batch_size}')

    flat = np.stack([member_oof[tag].astype(np.float32) for tag in tags], axis=0)
    flat = flat.reshape(len(tags), -1)
    labels = np.asarray(labels)
    best_f1 = -1.0
    best_weights = None
    checked = 0
    batch = []

    def evaluate(weight_rows):
        nonlocal best_f1, best_weights, checked
        weights = np.asarray(weight_rows, dtype=np.float32)
        scores = (weights @ flat).reshape(len(weights), len(labels), 3)
        preds = scores.argmax(axis=2)
        f1_values = _macro_f1_for_prediction_batch(preds, labels)
        idx = int(np.argmax(f1_values))
        checked += len(weights)
        if float(f1_values[idx]) > best_f1:
            best_f1 = float(f1_values[idx])
            best_weights = weights[idx].copy()

    for weights in product(grid, repeat=len(tags)):
        if not any(weights):
            continue
        batch.append(weights)
        if len(batch) >= batch_size:
            evaluate(batch)
            batch = []
    if batch:
        evaluate(batch)

    models = {tag: float(weight) for tag, weight in zip(tags, best_weights) if weight > 0}
    test_dist = None
    if all(tag in member_test for tag in models):
        test_sum = np.zeros_like(next(iter(member_test.values())), dtype=float)
        total_weight = 0.0
        for tag, weight in models.items():
            test_sum += member_test[tag] * weight
            total_weight += weight
        test_dist = np.bincount((test_sum / total_weight).argmax(axis=1), minlength=3).astype(int).tolist()

    regenerated = {
        'oof_f1_macro': best_f1,
        'n_folds': N_FOLDS,
        'optimisation': f'exhaustive OOF grid search over {checked} non-zero combinations; grid={list(grid)}',
        'models': models,
    }
    if test_dist is not None:
        regenerated['test_dist'] = test_dist

    ensemble_path.parent.mkdir(parents=True, exist_ok=True)
    ensemble_path.write_text(json.dumps(regenerated, indent=2), encoding='utf-8')
    print(f'Optimized ensemble OOF F1-macro: {best_f1:.4f}')
    print(f'Wrote regenerated weights to {ensemble_path}')
    return regenerated

print('Notebook-native training helpers defined.')


Notebook-native training helpers defined.


In [5]:

# Cell 6: Load-or-train all candidates, auto-select by OOF F1, optimize weights
#
# Fast path  (~2 min) : all caches present + ensemble JSON loaded in Cell 4.
# Rebuild path (hours): missing caches trained inline; missing JSON triggers
#   (1) individual OOF F1 ranking, (2) threshold filter, (3) weight grid search.

member_test = {}   # tag -> (2388, 3) test probabilities
member_oof  = {}   # tag -> (9543, 3) out-of-fold probabilities

def _load_member(tag):
    csv_p = Path(f'results/predictions/prob_test_{tag}.csv')
    npy_p = Path(f'results/predictions/test_proba_{tag}.npy')
    oof_n = Path(f'results/predictions/oof_proba_{tag}.npy')
    oof_c = Path(f'results/predictions/oof_proba_{tag}.csv')
    test_p = (pd.read_csv(csv_p)[['p0','p1','p2']].values if csv_p.exists()
              else np.load(npy_p) if npy_p.exists() else None)
    oof_p  = (np.load(oof_n) if oof_n.exists()
              else pd.read_csv(oof_c)[['p0','p1','p2']].values if oof_c.exists() else None)
    return (test_p.astype(np.float32) if test_p is not None else None,
            oof_p.astype(np.float32)  if oof_p  is not None else None)

# Step 1: load or train every candidate
for tag in CANDIDATE_TAGS:
    ptest, poof = _load_member(tag)
    if ptest is not None and poof is not None:
        member_test[tag], member_oof[tag] = ptest, poof
        print(f'[CACHED]      {tag}')
    elif tag in MODEL_CONFIGS:
        print(f'[CACHE MISS]  {tag} — training from scratch.')
        ptest, poof = train_member_from_scratch(tag, MODEL_CONFIGS[tag])
        member_test[tag], member_oof[tag] = ptest, poof
    else:
        raise FileNotFoundError(
            f'Cannot load or train {tag}: cache missing and no result JSON found.\n'
            f'Restore results/predictions/oof_proba_{tag}.npy or results/tables/{tag}_result.json.'
        )

# Step 2: rank every candidate by individual OOF F1-macro
individual_f1 = {
    tag: float(f1_score(y, member_oof[tag].argmax(1), average='macro'))
    for tag in CANDIDATE_TAGS if tag in member_oof
}
ranked = sorted(individual_f1.items(), key=lambda x: x[1], reverse=True)
print()
print(f'Individual OOF F1-macro (threshold = {MIN_OOF_F1}):')
for tag, f1 in ranked:
    mark = 'SELECTED' if f1 >= MIN_OOF_F1 else f'excluded  (< {MIN_OOF_F1})'
    print(f'  {f1:.4f}  {tag}  [{mark}]')

# Step 3: ensemble selection and weight optimization
if ENSEMBLE_WEIGHTS is None:
    # No pre-computed JSON: auto-select candidates above threshold, then optimize
    selected = [tag for tag, f1 in individual_f1.items() if f1 >= MIN_OOF_F1]
    if not selected:
        selected = [ranked[0][0]]
        print(f'WARNING: no candidates above {MIN_OOF_F1} — using best: {selected[0]}')
    print(f'\nAuto-selected {len(selected)}/{len(CANDIDATE_TAGS)} candidates for ensemble.')
    ENSEMBLE = optimize_ensemble_from_oof(member_oof, y, selected)
    ENSEMBLE_WEIGHTS = ENSEMBLE['models']
else:
    # Pre-computed JSON: validate all weighted members have probabilities
    missing = [t for t in ENSEMBLE_WEIGHTS if t not in member_test or t not in member_oof]
    assert not missing, f'Missing probabilities for weighted members: {missing}'
    print(f'\nUsing pre-optimized weights ({len(ENSEMBLE_WEIGHTS)} members from JSON).')

print()
print('Active ensemble weights:')
for tag, w in sorted(ENSEMBLE_WEIGHTS.items(), key=lambda x: -x[1]):
    print(f'  w={w:<5}  {tag}')


[CACHED]      finbert_fintwitter_10ep_fixtext
[CACHED]      finbert_fintwitter_7ep
[CACHED]      roberta_large_ts_v2
[CACHED]      debertav3_large_6ep
[CACHED]      finbert_fintwitter_10ep
[CACHED]      finbert_fintwitter
[CACHED]      debertav3_large_6ep_fixtext
[CACHED]      deberta_base_finance_fixtext

Individual OOF F1-macro (threshold = 0.87):
  0.9082  finbert_fintwitter_10ep_fixtext  [SELECTED]
  0.9056  finbert_fintwitter_10ep  [SELECTED]
  0.9049  finbert_fintwitter_7ep  [SELECTED]
  0.9025  finbert_fintwitter  [SELECTED]
  0.8943  roberta_large_ts_v2  [SELECTED]
  0.8889  debertav3_large_6ep_fixtext  [SELECTED]
  0.8882  debertav3_large_6ep  [SELECTED]
  0.8715  deberta_base_finance_fixtext  [SELECTED]

Using pre-optimized weights (8 members from JSON).

Active ensemble weights:
  w=1.0    finbert_fintwitter_10ep_fixtext
  w=1.0    finbert_fintwitter_7ep
  w=1.0    roberta_large_ts_v2
  w=1.0    debertav3_large_6ep
  w=0.75   finbert_fintwitter_10ep
  w=0.75   finbert_fintwi

In [6]:
# Cell 7: Honest out-of-fold performance of the weighted soft-vote ensemble
# Combines each member's leak-free OOF probabilities with the grid-searched weights.
oof_sum, wsum = np.zeros((len(y), 3)), 0.0
for tag, w in ENSEMBLE_WEIGHTS.items():
    if tag in member_oof:
        oof_sum += member_oof[tag] * w
        wsum += w
oof_ens  = oof_sum / wsum
oof_pred = oof_ens.argmax(1)
oof_f1   = f1_score(y, oof_pred, average='macro')

print(f'Ensemble OOF F1-macro : {oof_f1:.4f}   (weight total {wsum:.2f}, '
      f'{len(member_oof)} members)')
print(f'Accuracy     : {accuracy_score(y, oof_pred):.4f}')
print(f'Precision-mac: {precision_score(y, oof_pred, average="macro"):.4f}')
print(f'Recall-macro : {recall_score(y, oof_pred, average="macro"):.4f}')
print()
print(classification_report(y, oof_pred, target_names=['Bearish', 'Bullish', 'Neutral']))

Ensemble OOF F1-macro : 0.9201   (weight total 6.75, 8 members)
Accuracy     : 0.9357
Precision-mac: 0.9097
Recall-macro : 0.9315

              precision    recall  f1-score   support

     Bearish       0.87      0.92      0.90      1442
     Bullish       0.89      0.93      0.91      1923
     Neutral       0.97      0.94      0.95      6178

    accuracy                           0.94      9543
   macro avg       0.91      0.93      0.92      9543
weighted avg       0.94      0.94      0.94      9543



In [7]:
# Cell 7b: Weighted soft-vote on the test set (the final ensemble prediction)
test_sum, wsum_t = np.zeros((len(test_texts), 3)), 0.0
used = []
for tag, w in ENSEMBLE_WEIGHTS.items():
    if tag in member_test:
        test_sum += member_test[tag] * w
        wsum_t += w
        used.append(f'{tag} (w={w})')
test_proba_final = (test_sum / wsum_t).astype(np.float32)

print(f'Weighted soft-vote over {len(used)} encoders (total weight {wsum_t:.2f}):')
for u in used:
    print(f'  + {u}')
print(f'\nEnsemble OOF F1-macro: {oof_f1:.4f}')

Weighted soft-vote over 8 encoders (total weight 6.75):
  + finbert_fintwitter_10ep_fixtext (w=1.0)
  + finbert_fintwitter_7ep (w=1.0)
  + roberta_large_ts_v2 (w=1.0)
  + debertav3_large_6ep (w=1.0)
  + finbert_fintwitter_10ep (w=0.75)
  + finbert_fintwitter (w=0.75)
  + debertav3_large_6ep_fixtext (w=0.75)
  + deberta_base_finance_fixtext (w=0.5)

Ensemble OOF F1-macro: 0.9201


In [8]:
# Cell 8: Generate predictions and save pred_33.csv (with full validation)
test_pred  = test_proba_final.argmax(1)
submission = pd.DataFrame({'id': test['id'], 'label': test_pred.astype(int)})
submission.to_csv('pred_33.csv', index=False)

print(f'Predictions saved: pred_33.csv ({len(submission)} rows) - weighted soft-vote ensemble')
print('Distribution:')
print(submission['label'].value_counts().sort_index()
      .rename({0: 'Bearish', 1: 'Bullish', 2: 'Neutral'}))
print()

# Validate the deliverable
assert len(submission) == 2388,                        f'Expected 2388, got {len(submission)}'
assert list(submission.columns) == ['id', 'label'],    'Columns must be exactly [id, label]'
assert set(submission['label'].unique()) == {0, 1, 2}, f'Labels must be {{0,1,2}}'
assert submission['label'].isna().sum() == 0,          'NaN labels found!'
assert submission['id'].is_unique,                     'Duplicate ids found!'
assert (submission['label'] >= 0).all() and (submission['label'] <= 2).all(), 'Labels out of range'
min_class_pct = submission['label'].value_counts(normalize=True).min()
assert min_class_pct > 0.01, f'Under-represented class ({min_class_pct:.1%}) - possible model collapse'

print('All assertions PASSED.')
print()
print(submission.head(10).to_string(index=False))


Predictions saved: pred_33.csv (2388 rows) - weighted soft-vote ensemble
Distribution:
label
Bearish     385
Bullish     497
Neutral    1506
Name: count, dtype: int64

All assertions PASSED.

 id  label
  0      1
  1      2
  2      2
  3      1
  4      2
  5      1
  6      0
  7      0
  8      2
  9      2
